# CatBoost для долей восьми групп маршрутов

CPU. Группы фиксируются по среднему объёму маршрута до 20 сентября. Новый CatBoost прогнозирует долю каждой группы на всех восьми горизонтах по истории цели и шести статусов. Учится на трёх утренних стартах, использует причинные лаги и поправку к типичной доле группы в целевой слот. При сборке бленда доли маршрутов внутри групп остаются такими же, как у исходного прогноза, общий объём каждого горизонта сохраняется.

Сравниваются веса нового эксперта 0%, 25% и 50%. Вес выбирается только по двум неделям до test. Обе недели должны выиграть по среднему дневному score. Порог для конкурсного кандидата на локальном test составляет 0.0002 по средней дневной метрике без ухудшения общего score. Test уже изучался. Оценка получена до полного переобучения.

Если порог пройден, групповой CatBoost переобучается на всей доступной истории допустимых стартов. Доли маршрутов и объём сети берутся из уже готового родительского full-history сабмита. По умолчанию он лежит в group_share_parent.csv рядом с ноутбуком. Для явного указания пути используйте GROUP_SHARE_PARENT.

В метриках group_allocation_error показывает ошибку распределения между восемью группами при фиксированном объёме сети.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_group_share_cpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

DATA = ROOT/'blend_inputs/session6'
PARENT = Path(os.environ.get('GROUP_SHARE_PARENT', str(ROOT/'group_share_parent.csv')))
SIGNALS = ['target_1h']+[f'status_{i}' for i in range(1,7)]
STEP = pd.Timedelta(minutes=30)
GROUPS = np.arange(8,dtype=np.int64)
STARTS = (570,600,630)
MIN_TEST_GAIN = .0002


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def prepare_groups(path):
    raw = pd.read_parquet(path,columns=['route_id','timestamp',*SIGNALS])
    reference = raw.loc[raw.timestamp.lt('2025-09-20')].groupby('route_id').target_1h.mean().sort_index()
    if len(reference)!=1000:
        raise ValueError('Для разбиения нужны 1000 маршрутов')
    groups = pd.qcut(reference.rank(method='first'),8,labels=False).astype(np.int64).rename('group')
    grouped = raw.join(groups,on='route_id',validate='many_to_one')
    if grouped.group.isna().any() or grouped.duplicated(['timestamp','route_id']).any():
        raise ValueError('Некорректная панель')
    counts = grouped.groupby(['timestamp','group']).size()
    panel = grouped.groupby(['timestamp','group'])[SIGNALS].sum().sort_index()
    dates = pd.DatetimeIndex(panel.index.get_level_values('timestamp').unique())
    if (not dates.equals(pd.date_range(dates[0],dates[-1],freq=STEP))
            or len(panel)!=len(dates)*len(GROUPS) or not counts.eq(125).all()):
        raise ValueError('Неполная панель групп')
    values = panel.to_numpy(dtype=np.float64).reshape(len(dates),len(GROUPS),len(SIGNALS))
    if not np.isfinite(values).all() or np.any(values<0):
        raise ValueError('Некорректные значения')
    totals = values.sum(axis=1)
    shares = np.divide(values,totals[:,None,:],out=np.zeros_like(values),where=totals[:,None,:]>0)
    return groups,dates,shares,totals[:,0]

def group_features(origins,horizon):
    lead = origins+horizon
    lagged = (group_share[lead-48,:,0]+group_share[lead-336,:,0]+group_share[lead-672,:,0])/3
    prior = np.maximum(lagged,1e-5)
    now = group_share[origins]
    prev = group_share[origins-2]
    recent = (now+group_share[origins-1]+prev+group_share[origins-3])/4
    typical = (group_share[origins-48]+group_share[origins-336]+group_share[origins-672])/3
    shape = prior.shape
    matrix = np.concatenate([now,recent,now-prev,recent-typical,prior[...,None],
                             np.broadcast_to(GROUPS[None,:,None],shape+(1,)),
                             np.full(shape+(1,),horizon),
                             np.broadcast_to(minutes[origins,None,None],shape+(1,)),
                             np.broadcast_to(weekday[origins,None,None],shape+(1,))],axis=2)
    return matrix.reshape(len(origins)*len(GROUPS),-1).astype(np.float32),prior.ravel()

def fit_groups(boundary):
    stop = dates.searchsorted(pd.Timestamp(boundary),side='left')
    eligible = np.flatnonzero(np.isin(minutes,STARTS)&(np.arange(len(dates))>=672))
    features,targets,weights = [],[],[]
    for h in range(1,9):
        origins = eligible[eligible+h<stop]
        lead = origins+h
        origins = origins[network_target[lead]>0]
        lead = origins+h
        matrix,prior = group_features(origins,h)
        features.append(matrix)
        targets.append(group_share[lead,:,0].ravel()/prior)
        weights.append(prior*np.repeat(network_target[lead],len(GROUPS)))
    weight = np.concatenate(weights)
    model = CatBoostRegressor(iterations=320,depth=4,learning_rate=.045,loss_function='MAE',
                              l2_leaf_reg=20,thread_count=4,random_seed=SEED,verbose=False,
                              task_type='CPU',allow_writing_files=False)
    model.fit(np.concatenate(features),np.concatenate(targets),sample_weight=weight/weight.mean())
    print('Групповой CatBoost',boundary,'строк',len(weight),flush=True)
    return model

def forecast_groups(model,cutoffs):
    cutoff = pd.DatetimeIndex(cutoffs).unique()
    origins = dates.get_indexer(cutoff)
    if np.any(origins<672):
        raise ValueError('Нет причинной истории')
    blocks = []
    for h in range(1,9):
        x,prior = group_features(origins,h)
        forecast = np.maximum(model.predict(x),0)*prior
        forecast = forecast.reshape(len(origins),len(GROUPS))
        if not np.isfinite(forecast).all() or np.any(forecast.sum(axis=1)<=0):
            raise ValueError('Некорректные групповые прогнозы')
        forecast /= forecast.sum(axis=1,keepdims=True)
        blocks.append(pd.DataFrame(dict(cutoff=cutoff.repeat(len(GROUPS)),horizon=h,
                                        group=np.tile(GROUPS,len(origins)),group_share=forecast.ravel())))
    return pd.concat(blocks,ignore_index=True)

def replace_group_shares(frame,baseline,forecast,weight):
    keys = GROUP_KEYS+['group']
    table = frame[KEYS].assign(group=frame.route_id.map(route_group),baseline=np.asarray(baseline,dtype=np.float64))
    totals = table.groupby(keys).baseline.transform('sum')
    network = table.groupby(GROUP_KEYS).baseline.transform('sum')
    joined = table[keys].merge(forecast,on=keys,how='left',validate='many_to_one',sort=False)
    if table.group.isna().any() or joined.group_share.isna().any() or totals.le(0).any():
        raise ValueError('Некорректные ключи групп или объёмы')
    predicted = table.baseline.to_numpy()/totals.to_numpy()*network.to_numpy()*joined.group_share.to_numpy()
    return with_volume(frame,(1-weight)*table.baseline.to_numpy()+weight*predicted,anchor=table.baseline.to_numpy())

def evaluate(frame,base,model,weights,period,fold):
    group_forecast = forecast_groups(model,frame.cutoff.unique())
    rows = []
    y = frame.y.to_numpy(dtype=np.float64)
    group_keys = GROUP_KEYS+['group']
    for weight in weights:
        prediction = replace_group_shares(frame,base,group_forecast,weight)
        by_group = frame[GROUP_KEYS].assign(group=frame.route_id.map(route_group),actual=y,prediction=prediction)
        by_group = by_group.groupby(group_keys)[['actual','prediction']].sum()
        network = by_group.groupby(level=GROUP_KEYS).transform('sum')
        matched = by_group.actual * network.prediction / network.actual
        group_error = (by_group.prediction-matched).abs().sum()/network.prediction.groupby(level=GROUP_KEYS).first().sum()
        daily = np.asarray([metric(g.y,prediction[g.index])['score'] for _,g in frame.groupby('cutoff')])
        rows.append(dict(period=period,fold=fold,weight=weight,**metric(frame.y,prediction),
                         daily_mean=float(daily.mean()),daily_p90=float(np.quantile(daily,.9)),
                         group_allocation_error=float(group_error)))
    return rows


In [ ]:
route_group,dates,group_share,network_target = prepare_groups(DATA/'train_solo_track.parquet')
minutes = (dates.hour*60+dates.minute).to_numpy()
weekday = dates.day_of_week.to_numpy()
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
weights = [0.,.25,.5]
rows = []
for start,end in [('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[(oof.cutoff>=start)&(oof.cutoff<end)].reset_index(drop=True)
    meta = fit_meta(train)
    model = fit_groups(start)
    for period,frame in [('train_oof',train),('val',val)]:
        rows.extend(evaluate(frame,predict_meta(frame,meta),model,weights,period,start))
    display(pd.DataFrame(rows).loc[lambda t:t.fold.eq(start)])
validation = pd.DataFrame(rows)
ranking = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='weight',values='daily_mean')
eligible = [weight for weight in weights[1:] if ranking[weight].lt(ranking[0.]).all()]
winner = min(eligible,key=lambda w:(ranking[w].mean(),w)) if eligible else 0.
validation.to_csv(OUT/'validation_metrics.csv',index=False)
(OUT/'frozen_selection.json').write_text(json.dumps(dict(weight=winner,min_test_gain=MIN_TEST_GAIN,test_used_for_weight_selection=False),indent=2))
print('Выбранный вес',winner,flush=True)


In [ ]:
test = read_panel(TEST_PATH,'test_predictions.parquet','2025-10-18','2025-11-01')
meta = fit_meta(oof)
model = fit_groups('2025-10-18')
for period,frame in [('train_oof',oof),('test',test)]:
    rows.extend(evaluate(frame,predict_meta(frame,meta),model,list(dict.fromkeys([0.,winner,1.])),period,'2025-10-18'))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
r = metrics.loc[metrics.period.eq('test')].set_index('weight')
gain = float(r.loc[0.,'daily_mean']-r.loc[winner,'daily_mean'])
promoted = bool(winner>0 and gain>=MIN_TEST_GAIN and r.loc[winner,'score']<=r.loc[0.,'score'])
(OUT/'decision.json').write_text(json.dumps(dict(promoted=promoted,weight=winner,daily_gain=gain,
                                                 test_before_full_refit=True),indent=2))
print('Прошёл порог для сабмита:',promoted,flush=True)


In [ ]:
if promoted and PARENT.is_file():
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    if (list(parent.columns)!=['id','y_pred'] or len(parent)!=8000 or len(request)!=8000
            or not parent.id.is_unique or not request.id.is_unique or set(parent.id)!=set(request.id)
            or not np.isfinite(parent.y_pred).all() or parent.y_pred.lt(0).any()):
        raise ValueError('Некорректный родительский сабмит')
    cutoff = request.timestamp.min()-STEP
    if dates[-1]!=cutoff:
        raise ValueError('Не совпала граница истории')
    final = fit_groups(cutoff+STEP)
    query = request.merge(parent,on='id',validate='one_to_one',sort=False)
    query['cutoff'] = cutoff
    query['horizon'] = ((query.timestamp-cutoff)/STEP).astype(int)
    prediction = replace_group_shares(query,query.y_pred,forecast_groups(final,[cutoff]),winner)
    if not query.id.equals(request.id) or not np.isfinite(prediction).all() or np.any(prediction<0):
        raise ValueError('Некорректный результат')
    destination = OUT/'submission_group_share_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    final.save_model(str(OUT/'group_share_full_history.cbm'))
    print('Сабмит:',destination,flush=True)
elif promoted:
    print('Загрузите родительский CSV:',PARENT,flush=True)
print('Результаты:',OUT,flush=True)
